# Thesis Mechanism Spine — living working notebook

**Locked 2026-06-17.** Working notebook for the MSc thesis (defence June 2027).
Commentary + reproducible code for both arms. Edit as the work evolves.

## Thesis claim (falsifiable)
Quantum entanglement improves a biomedical learning task **only when the data carries
matching multi-channel joint structure** (Huang et al. 2021; Kübler et al. 2021).
Tested on two arms sharing one ablation methodology — *remove the entanglement, measure
what is lost.* Extension/application study, not a new method (paper rule #1). Expected
result on the advantage axis is null-or-marginal; the contribution is the **predictive
principle**, not a supremacy claim.

- **Arm 1 — entanglement in the ENCODING** (consolidation of existing per-scheme results).
- **Arm 2 — entanglement in the INITIALIZATION** (new bounded swing; circuit as a fixed
  sampler → no barren plateau). Deciding control = classical-distribution-matched-to-
  quantum-marginals.

Full plan + timeline + kill dates: `thesis_spine_2026.md`. Phase 0 findings:
`arm1_substrate_phase0.md`.

## Arm 1 — Phase 0 substrate

Source of truth: `_perscheme_results.json` (per-scheme, **single run, NO seeds/CI**) and
`_ecg_rawbaseline.json` (ECG raw ROCKET baseline, 5 seeds). All multivariate datasets are
3-channel. Δ_ent = (best entangling single-scheme acc) − (Sep separable acc).

In [ ]:
import json
from pathlib import Path
import pandas as pd

BASE = Path('.')  # run from /Users/eldana/Documents/Quantum/Thesis/QIP
per = json.loads((BASE / '_perscheme_results.json').read_text())

# Entanglement ordering CONFIRMED 2026-06-17:
#   Sep < CRyE < CSE < GBE < PA-CSE < CP-2L
# (PA-CSE = pixel-adaptive CSE; image/ECG only, not in this multivariate set.)
SEP = 'Sep'                                    # separable / no-entanglement baseline
ENTANGLING = ['CRyE', 'CSE', 'GBE', 'CP-2L']   # least → most entangling (present here)

rows = []
for ds, d in per.items():
    s = d['schemes']
    sep_acc = s[SEP]['acc']
    ent = {k: s[k]['acc'] for k in ENTANGLING if k in s}
    best = max(ent, key=ent.get)
    rows.append({
        'dataset': ds,
        'K': d['K'],
        'raw': round(d['raw'], 4),
        'Sep': round(sep_acc, 4),
        'best_ent_scheme': best,
        'best_ent_acc': round(ent[best], 4),
        'delta_ent': round(ent[best] - sep_acc, 4),
        'headroom': round(1 - d['raw'], 4),
        # per-scheme accs in entanglement order, to inspect monotonicity / U-shape
        **{k: round(s[k]['acc'], 4) for k in ENTANGLING if k in s},
    })

df = pd.DataFrame(rows).sort_values('delta_ent', ascending=False).reset_index(drop=True)
df

### Adversarial findings (read before building the metric)

1. **The "entanglement helps UWave, fails ECG" headline is FALSE on canonical splits.**
   On UWave per-scheme, **Sep (separable) is the best single scheme** (0.9125); the
   entangling variants are at or below it (Δ = −0.003). The 0.976 UWave "positive" was a
   cross-scheme **ensemble** on an expanded split — not single-scheme entanglement. Do not
   build the thesis on the ensemble number and call it an entanglement effect.
2. **Every Δ_ent is tiny and has NO error bars** (+0.027 to −0.008, single runs). Plausibly
   within seed noise. **RIGOR GATE:** re-run per-scheme with ≥5 seeds + CIs before asserting
   any entanglement effect.
3. **Confound to break:** the biggest Δ (Handwriting, x/y/force) is also the dataset with the
   most accuracy headroom. "Cross-channel coupling helps" and "there was just room to improve"
   are tangled. Arm 1's job: show a coupling metric *C* predicts Δ_ent **after controlling for
   headroom *H***. If *C* adds nothing over *H*, there is no entanglement-structure result
   (still a finding).

**Revised Arm 1 hypothesis:** Δ_ent = f(cross-channel coupling *C*, headroom *H*). Single-channel
ECG (*C*=0) anchors the low end — the observed entanglement-null becomes a *confirming* point.

In [ ]:
# ECG raw ROCKET baseline (PhysioNet2017, single lead) — zero-coupling anchor.
ecg = json.loads((BASE / '_ecg_rawbaseline.json').read_text())
f1s = [v['macro_f1_nao'] for v in ecg['seeds'].values()]
print(f"ECG raw ROCKET macro-F1: mean={sum(f1s)/len(f1s):.4f}  n_seeds={len(f1s)}")
print(f"config: {ecg['config']}")

# ECG QPIE per-scheme TRACED from paper_aeon/main.tex (5 seeds, 1D CNN head, Tab phase1b):
#   Sep 0.669 | PA-CSE 0.670 (tied best) | GBE 0.598 (worst — 7-pt entanglement penalty).
#   Paper: "No entangling scheme beats the separable baseline (Sep) on any head."
#   -> zero-coupling null anchor CONFIRMED.
# Raw ROCKET (~0.72) > QPIE-CNN best (0.67) -> encoding is lossy: no ABSOLUTE advantage,
#   consistent with mechanism-not-supremacy spine.

## Phase 0 — CLOSED (2026-06-17)

- [x] Scheme entanglement ordering CONFIRMED: Sep < CRyE < CSE < GBE < PA-CSE < CP-2L.
- [x] Image-domain dose-response CONFIRMED on PathMNIST (5 seeds, paired t-tests): CSE helps*,
      PA-CSE hurts*. CSE→PA-CSE within-family sign flip = keystone.
- [x] ECG null anchor traced from `paper_aeon/main.tex` (Sep 0.669 / PA-CSE 0.670 / GBE 0.598;
      no entangler beats Sep). Raw ROCKET ~0.72 > QPIE 0.67 → encoding lossy, no absolute advantage.
- [x] DECISION: multivariate per-scheme DEMOTED to honest supporting breadth (single-run,
      exploratory). PathMNIST + ECG carry the thesis. No 5-seed re-run. → proceed to Arm 2.

# Arm 2 — PRE-REGISTRATION (locked 2026-06-17, before any model run)

**Plain-English question.** Training is a hiker walking downhill to the lowest valley; the
initial weights are *where we drop the hiker*. Does dropping it at a point whose weights carry
relationships **taken from the ECG data's own structure** ("data-aware entangled init") get it
down faster / to a lower valley / into a flatter (more robust) valley than standard recipes and
than controls that strip either the relationships or the data-matching?

**Locked design rules**
- Only the **first hidden layer's initial weights** vary across conditions. Architecture, ECG
  data + split, optimizer, LR, batch size, epoch budget, and the set of random seeds are
  IDENTICAL across all conditions. Any outcome difference is attributable to init alone.
- The quantum circuit is a **FIXED SAMPLER** (run once to produce numbers, never trained) →
  no barren plateau.
- ECG representation: reuse the Phase 8 pipeline (talks to Arm 1's null anchor; zero new plumbing).
- 5 seeds (matches portfolio rigor: Paper 1 + ECG both 5-seed).

**Conditions (first-hidden-layer init only)**

| # | condition | plain meaning | role |
|---|-----------|---------------|------|
| 1 | He        | textbook recipe | classical SOTA baseline |
| 2 | Xavier    | textbook recipe | classical SOTA baseline |
| 3 | Orthogonal| textbook recipe | classical SOTA baseline |
| 4 | Sep       | circuit, no entanglement → independent weights | data-blind floor |
| 5 | Data-blind entangled (CRyE/GBE-style) | relationships, but arbitrary (not from data) | control: unmatched entanglement |
| 6 | **Data-aware entangled (CSE-style)** | relationships set by ECG feature-channel correlations | **HEADLINE candidate** |
| 7 | Marginal-matched | copy #6's per-weight spread, roll each weight independently (relationships erased) | **decider control** |

**Pre-registered hypotheses**
- **H1 (speed):** #6 reaches the accuracy threshold in fewer epochs than #1 (He).
- **H2 (decisive — do relationships matter?):** #6 beats #7. If not → correlations are inert,
  the "quantum" was just a fancy spread. PRIMARY decider.
- **H3 (does data-matching matter?):** #6 beats #5. If not → arbitrary correlations work as well,
  so it is not about matching the data.
- **H4 (landscape):** #6 lands in a flatter minimum (lower Hessian trace) than #1.

**Metrics (pre-registered)**
- epochs-to-threshold (threshold = fixed macro-F1, set from the He baseline curve)
- final test macro-F1 / accuracy
- loss-landscape sharpness at the solution: Hessian trace (or top eigenvalue / SAM-style)
- across-seed variance (5 seeds)

**Decision rule / KILL CRITERION (~mid-Jul)**
- Primary = H2. If #6 does NOT beat #7 within seed noise across 5 seeds → declare the
  entanglement-correlation **null for initialization**, freeze Arm 2, write it up as a clean
  negative (consistent with the spine: unmatched/arbitrary entanglement does not help, and here
  even data-matched correlations do not transfer to init). Do NOT expand chasing a win.
- If #6 beats #7 AND #6 beats #5 → entanglement-from-data is the active ingredient. Expand
  (more layers, a second dataset) as time before defence allows.

**Open details to pin (not blocking the pre-registration)**
- Exact data property #6 reads — proposed: pairwise correlation matrix of the ECG feature
  channels (same quantity CSE uses in Arm 1).
- First-hidden-layer width H; accuracy threshold value (read off He baseline).

## Open items

- [ ] **Confirm scheme entanglement ordering** (Sep / CRyE / GBE / CP-2L / CSE) — blocks the metric.
- [ ] Trace ECG QPIE per-scheme numbers from `Phase8_PhysioNet2017_ECG.ipynb`.
- [ ] Re-run multivariate per-scheme with ≥5 seeds + CIs (rigor gate).
- [ ] Define coupling metric *C*; regress Δ_ent on *C* controlling for *H*.

## Arm 2 — initialization (placeholder; build in Phase 1, kill date ~mid-Jul)

6 pre-registered init conditions: He, Xavier, Orthogonal, Entangled-circuit, Product-circuit
(ablation), **Classical-matched-to-quantum-marginals (deciding control)**. Metrics:
epochs-to-threshold, final test acc, loss-landscape sharpness (Hessian-trace), across-seed
variance. Dataset: ECG features (reuse Phase 8 pipeline). Decisive comparison =
entangled vs marginal-matched.